In [9]:
import pyspark
from pyspark.sql import SparkSession

In [10]:
spark = SparkSession \
    .builder \
    .master("local") \
    .appName('jupyter-pyspark') \
        .config("hive.metastore.uris", 
                "thrift://hive-metastore:9083") \
        .enableHiveSupport() \
    .getOrCreate()

In [11]:
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print('howdy')

howdy


# HDFS

http://localhost:50070/  (Outside the docker environment)

In [12]:
# Read local
df = spark.read.csv("/home/jovyan/datasets/customers/customers.csv", header=True)
df.sample(0.2).toPandas()

,First,Last,Email,Gender,Last IP Address,City,State,Total Orders,Total Purchased,Months Customer
0,Arial,Photo,aphoto@dayrep.com,F,24.0.14.56,Newark,NJ,1,680,1
1,Carol,Ling,cling@superrito.com,F,23.180.242.66,Syracuse,NY,2,440,6
2,Dan,Delyons,ddelyons@dayrep.com,M,24.38.224.161,Greenwich,CT,2,2570,10
3,Erin,Detyers,edetyers@dayrep.com,F,70.209.14.54,Tampa,FL,5,1105,38


In [13]:
# Write to HDFS
df.write.csv("webhdfs://namenode:50070/user/demo/customers/")

AnalysisException: [PATH_ALREADY_EXISTS] Path webhdfs://namenode:50070/user/demo/customers already exists. Set mode as "overwrite" to overwrite the existing path.

In [14]:
# Read back from HDFS
spark.read.csv("webhdfs://namenode:50070/user/demo/customers/", header=False).show()

+------+----------+--------------------+---+---------------+-----------+---+---+----+---+
|   _c0|       _c1|                 _c2|_c3|            _c4|        _c5|_c6|_c7| _c8|_c9|
+------+----------+--------------------+---+---------------+-----------+---+---+----+---+
|    Al|    Fresco|  afresco@dayrep.com|  M|  74.111.18.161|   Syracuse| NY|  1|  45|  1|
|  Abby|      Kuss|     akuss@rhyta.com|  F|  23.80.125.101|    Phoenix| AZ|  1|  25|  2|
| Arial|     Photo|   aphoto@dayrep.com|  F|     24.0.14.56|     Newark| NJ|  1| 680|  1|
| Bette|     Alott|    balott@rhyta.com|  F| 56.216.127.219|    Raleigh| NC|  6| 560| 18|
|  Barb|    Barion|bbarion@superrito...|  F|   38.68.15.223|     Dallas| TX|  4|1590|  1|
| Barry|DeHatchett|bdehatchett@dayre...|  M|  23.192.215.78|     Boston| MA|  1|  15|  6|
|  Bill|   Melator| bmelator@einrot.com|  M|   24.11.125.10|       Orem| UT|  9|6090| 35|
| Candi|     Cayne|    ccayne@rhyta.com|  F|    24.39.14.15|   Portland| ME|  1| 620|  2|
| Carol|  

# Hive

In [15]:
spark.sql("show tables;").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [16]:
spark.sql("use labc;")
spark.sql("show tables;").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     labc|customers|      false|
|     labc|marketing|      false|
|     labc|  surveys|      false|
|     labc|   tweets|      false|
+---------+---------+-----------+



## Internal Tables

Internal tables are stored in `/user/hive/warehouse`

In [12]:
# read Local Data
df = spark.read.option("multiline","true").json("/home/jovyan/datasets/json-samples/stocks.json")
df.toPandas()

,price,symbol
0,126.82,AAPL
1,3098.12,AMZN
2,251.11,FB
3,1725.05,GOOG
4,128.39,IBM
5,212.55,MSFT
6,78.00,NET
7,497.00,NFLX
8,823.80,TSLA
9,45.11,TWTR


In [13]:
# Hive Internal Table Bulk import

spark.sql("DROP TABLE IF EXISTS default.stocks;")
df.createOrReplaceTempView("tmp_stocks") 
spark.sql("""
CREATE TABLE IF NOT EXISTS default.stocks 
    AS select * from tmp_stocks
""")
spark.sql("select * from default.stocks").show()

+-------+------+
|  price|symbol|
+-------+------+
| 126.82|  AAPL|
|3098.12|  AMZN|
| 251.11|    FB|
|1725.05|  GOOG|
| 128.39|   IBM|
| 212.55|  MSFT|
|   78.0|   NET|
|  497.0|  NFLX|
|  823.8|  TSLA|
|  45.11|  TWTR|
+-------+------+



In [14]:
# Hive Internal Table create stuff.

spark.sql("""drop table if exists default.department""")
spark.sql("""CREATE TABLE default.department(
department_id int ,
department_name string
)    
""")
spark.sql("""
INSERT INTO default.department values (101,"Oncology")    
""")
spark.sql("""
INSERT INTO default.department values (102,"Hematology")    
""")
spark.sql("SELECT * FROM default.department").show()

+-------------+---------------+
|department_id|department_name|
+-------------+---------------+
|          101|       Oncology|
|          102|     Hematology|
+-------------+---------------+



## External Tables

External tables exist in the metastore only and point to an HDFS loocation

In [15]:
# Create a database
spark.sql("CREATE DATABASE IF NOT EXISTS ischool")
spark.sql("show databases;").show()

+---------+
|namespace|
+---------+
|  default|
|  ischool|
+---------+



In [16]:
# external table
spark.sql("drop table if exists default.grades")
spark.sql("""
create external table default.grades (
  year int,
  semester string,
  course string,
  credits int,
  grade string
) 
row format delimited 
fields terminated by '\t' 
location  'hdfs:///user/root/grades/*.tsv'
""")
spark.sql("select * from default.grades").show()

+----+--------+------+-------+-----+
|year|semester|course|credits|grade|
+----+--------+------+-------+-----+
+----+--------+------+-------+-----+



In [17]:
spark.sql("SELECT * FROM default.department").show()

+-------------+---------------+
|department_id|department_name|
+-------------+---------------+
|          101|       Oncology|
|          102|     Hematology|
+-------------+---------------+



In [18]:
spark.sql("select * from ischool.grades").show()

+----+--------+------+-------+-----+
|year|semester|course|credits|grade|
+----+--------+------+-------+-----+
|2016|    Fall|IST346|      3|    A|
|2016|    Fall|CHE111|      4|   A-|
|2016|    Fall|PSY120|      3|   B+|
|2016|    Fall|IST256|      3|    A|
|2016|    Fall|ENG121|      3|   B+|
|2015|    Fall|IST101|      1|    A|
|2015|    Fall|IST195|      3|    A|
|2015|    Fall|IST233|      3|   B+|
|2015|    Fall|SOC101|      3|   A-|
|2015|    Fall|MAT221|      3|    C|
|2016|  Spring|GEO110|      3|   B+|
|2016|  Spring|MAT222|      3|    A|
|2016|  Spring|SOC121|      3|   C+|
|2016|  Spring|BIO240|      3|   B-|
|2017|  Spring|IST462|      3|    A|
|2017|  Spring|MAT411|      3|    C|
|2017|  Spring|SOC422|      3|   B-|
|2017|  Spring|ENV201|      3|   A-|
+----+--------+------+-------+-----+

